# Procurement Intelligence Assistant — MVP

RAG-based chatbot answering procurement, sourcing, and supply chain questions
from a curated set of YouTube video transcripts.

*Pipeline stages:* transcription (done) → chunking → embeddings → vector store → retrieval + LLM

## 1. Setup

In [1]:
# Core dependencies for chunking and token counting
import json
import tiktoken

# cl100k_base is the tokenizer used by GPT-3.5/4 family models —
# a reasonable proxy for token count even if using a different embedding model
enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) ->int:
    return len(enc.encode(text))

## 2. Load Transcripts


#Load transcripts

Sourcedata: data/transcripts.json
Structure: {video_id: {title, segments: [{start, end, text}, ...]}}
#Two videos (uSjrTpJHn2g, U1E6rtnreoA) were pre-trimmed to remove webinar preamble/roll-call content before this file was saved.

In [2]:
with open("data/transcripts.json") as f:
    data = json.load(f)

print(f"Loaded {len(data)} videos")

Loaded 10 videos


## 3. Chunking

*Strategy:* merge consecutive Whisper segments into ~250-token chunks,
with ~40-token overlap so ideas that span a chunk boundary (e.g. a numbered
list) still appear complete in at least one chunk.

Each chunk keeps video_id, title, start, end as metadata —
needed later for citations and clickable timestamped YouTube links.

In [3]:
def count_tokens(text: str) -> int:
    """Return the token count of a string using the cl100k_base tokenizer."""
    return len(enc.encode(text))


def chunk_segments(segments, target_tokens=250, overlap_tokens=40):
    """
    Merge a list of {start, end, text} segments into token-bounded chunks.

    Args:
        segments: list of transcript segments for one video
        target_tokens: approx. token count to accumulate before closing a chunk
        overlap_tokens: approx. token count carried over into the next chunk

    Returns:
        list of {text, start, end, token_count} dicts
    """
    chunks = []
    current = []
    current_tokens = 0

    for seg in segments:
        current.append(seg)
        current_tokens += count_tokens(seg["text"])

        if current_tokens >= target_tokens:
            chunks.append(_build_chunk(current, current_tokens))
            current, current_tokens = _take_overlap(current, overlap_tokens)

    if current:
        chunks.append(_build_chunk(current, current_tokens))

    return chunks


def _build_chunk(seg_list, token_count):
    """Join a list of segments into a single chunk record."""
    return {
        "text": " ".join(s["text"].strip() for s in seg_list),
        "start": seg_list[0]["start"],
        "end": seg_list[-1]["end"],
        "token_count": token_count,
    }


def _take_overlap(seg_list, overlap_tokens):
    """Return the trailing segments (and their token count) to seed the next chunk."""
    overlap_seg_list = []
    overlap_count = 0
    for s in reversed(seg_list):
        overlap_count += count_tokens(s["text"])
        overlap_seg_list.insert(0, s)
        if overlap_count >= overlap_tokens:
            break
    return overlap_seg_list, overlap_count

## 4.Test chunk size in one video

In [4]:
test_video = "ZCIJQJRw6xw"

for target in [200, 250, 300]:
    test_chunks = chunk_segments(
        data[test_video]["segments"],
        target_tokens=target,
        overlap_tokens=int(target * 0.15)
    )
    print(f"\n=== target={target} tokens ({len(test_chunks)} chunks) ===")
    print(test_chunks[1]["text"])


=== target=200 tokens (13 chunks) ===
Whether it's software for a startup or machines for a factory, procurement decides who delivers what, when, and how much it costs. For example, think of procurement like planning a big wedding. You don't just go buy food and flowers. You plan your guest list, find reliable caterers, compare quotations, ensure everything arrives on time, track the budget, sign contracts. That's procurement on a business scale. Let's take a real-world example. Tata Motors, one of India's largest automobile manufacturers. To build each vehicle, they need tires from Bridgestone, steel from Tata Steel, electronics from Bosch, paint, seats, dashboards, software systems, etc. Tata Motors doesn't manufacture all of this in-house. Instead, they procure parts from a global network of suppliers who specialize in those products. If even one part, like a microchip, doesn't arrive on time, production halts, deadlines are missed, and millions are lost. That's how powerful procur

## 5. Lock in final size and run on all videos

In [5]:
FINAL_TARGET = 250   # <- update based on Step 4
FINAL_OVERLAP = 40    # <- update based on Step 4

all_chunks = []

for vid, content in data.items():
    video_chunks = chunk_segments(
        content["segments"],
        target_tokens=FINAL_TARGET,
        overlap_tokens=FINAL_OVERLAP
    )
    for c in video_chunks:
        c["video_id"] = vid
        c["title"] = content["title"]
        all_chunks.append(c)

print(f"Total chunks: {len(all_chunks)}")
print(f"Videos processed: {len(data)}")
print(f"Avg chunks per video: {len(all_chunks) / len(data):.1f}")

Total chunks: 160
Videos processed: 10
Avg chunks per video: 16.0


## 6. Sanity check the full set

In [6]:
import random

sample_chunks = random.sample(all_chunks, 5)

for c in sample_chunks:
    print(f"--- {c['title']} ({c['start']:.0f}s-{c['end']:.0f}s, {c['token_count']} tokens) ---")
    print(c["text"])
    print()

--- Supply Chain Risk Management Strategy | Response to Risk | Contingency Plan (398s-490s, 250 tokens) ---
Once a risk event occurs, you can use your action plan to decide how to handle the relevant risk. Depending on whether the level of impact is high or low, action plan with appropriate effect should be chosen. Let's take an example. Consider a case where a supplier's production plant is located in an area with a high earthquake risk. What should your framework for preventive action be? For a low-impact area, it may be reasonable to decide that the risk of an earthquake hitting the supplier site is accepted. In this case, no action needed need be defined. For a high-impact area, on the other hand, you may want to define some action. For example, establishing an alternative source of supply or taking out CBI, which is contingent business interruption insurance. To implement action planning on a sustainable basis and to ensure long-term success, support by top-level management is imp

## 7.Save the chunks

In [7]:
with open("data/chunks.json", "w") as f:
    json.dump(all_chunks, f, indent=2)

print("Saved", len(all_chunks), "chunks to data/chunks.json")

Saved 160 chunks to data/chunks.json


## 8. Install dependencies for Embeddings

In [8]:
!pip install chromadb openai tiktoken


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 9. Setup

In [9]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
from dotenv import load_dotenv
load_dotenv()

import json
import chromadb
from openai import OpenAI

client = OpenAI()  # now picks up OPENAI_API_KEY from .env
chroma_client = chromadb.PersistentClient(path="./chroma_db")

## 10.Loading chunks

Source: data/chunks.json
Each chunk carries: text, start, end, video_id, title, token_count

In [11]:
with open("data/chunks.json") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 160 chunks


## 11. Create Chroma collection

In [12]:
collection = chroma_client.get_or_create_collection(
    name="procurement_scm",
    metadata={"hnsw:space": "cosine"}
)

## 12. Embed and upsert chunks

Each chunk gets embedded and stored with its metadata (video_id, title,
timestamps) so retrieved results can be traced back to source + citation.

In [13]:
def embed_text(text: str) -> list[float]:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding


batch_size = 50

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    ids = [f"{c['video_id']}_{i+j}" for j, c in enumerate(batch)]
    texts = [c["text"] for c in batch]
    embeddings = [embed_text(t) for t in texts]
    metadatas = [
        {
            "video_id": c["video_id"],
            "title": c["title"],
            "start": c["start"],
            "end": c["end"]
        }
        for c in batch
    ]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas
    )

    print(f"Upserted {i + len(batch)}/{len(chunks)}")

print("Done embedding and indexing.")

Upserted 50/160
Upserted 100/160
Upserted 150/160
Upserted 160/160
Done embedding and indexing.


## 13. Sanity check — test a retrieval query

In [14]:
def test_query(query, n_results=3):
    results = collection.query(
        query_embeddings=[embed_text(query)],
        n_results=n_results
    )
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        print(f"--- {meta['title']} ({meta['start']:.0f}s) — dist={dist:.3f} ---")
        print(doc)
        print()

test_query("what are the stages of contract management?")

--- Contract Management in Procurement | Stages & Tools (59s) — dist=0.250 ---
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM platforms for tracking execution. Stage 3. Contract monitoring and com

In [15]:
test_query("how do you segment suppliers by importance?")

--- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (1870s) — dist=0.391 ---
From that, well, then you can be able to understand how do you manage that particular supplier in that particular item of your portfolio? What we believe is important is that you're not just focusing on segmenting your suppliers, but that you're automating that segmentation. Technology can help to guide that. But of course, the parameters need to be set by the business together with your organization, in your teams, and of course, aligned with your strategic goals. This automated segmentation, though, is a rule based setup that we do together with our customers. And you should be able to, of course, ascertain this with any technology that you would be finding out there on the market for supplier relationship management. It's a kind of if-then methodology, which is then allowing our customers to quickly segment their supplier bases, approve them, not approve them, but also then b

In [16]:
test_query("what causes stockouts and how is inventory tracked?")

--- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (176s) — dist=0.487 ---
Inventory management can be the heartbeat but of a supply chain or a company and its supply chain, especially because it plays a very important role in optimizing that all the different various aspects of a supply chain and its operations run as smoothly as possible and can actually function. So now we're going to take a look at some of these operations or what is within inventory management. Now look at those next. Now some of the aspects we're going to talk about is balancing supply and demand. I mentioned that a little bit earlier, but we're going to talk about it here too. Inventory management is going to be all about finding the perfect balance between supply and demand. Maintaining the right level of inventory, companies or organizations can ensure that the products are readily available when customers need them. This reduces stockouts, which can lead to lost sales opportuniti

## 13.5 Wrap retrieval as a LangChain Tool

Formalizes the existing retrieval logic as a Tool so it can be handed to
a LangChain agent later, which will decide per-question whether to use
this, a general-knowledge tool, or both.

In [ ]:
from langchain.tools import Tool
import re
import json as _json

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    BM25Okapi = None  # hybrid search degrades to embeddings-only if not installed

try:
    import cohere
except ImportError:
    cohere = None  # reranking is skipped if not installed / no API key

RELEVANCE_THRESHOLD = 0.45  # cosine distance fallback — must match app.py / langsmith_eval.py
RERANK_THRESHOLD = 0.3      # Cohere relevance score (0-1), used when reranking is configured
EMBED_TOP_K = 15
BM25_TOP_K = 15
FINAL_TOP_K = 6

with open("data/chunks.json") as f:
    _all_chunks = _json.load(f)

_bm25 = None
if BM25Okapi is not None and _all_chunks:
    _bm25_tokenized = [re.findall(r"\w+", c["text"].lower()) for c in _all_chunks]
    _bm25 = BM25Okapi(_bm25_tokenized)

_cohere_client = None
if cohere is not None and __import__("os").environ.get("COHERE_API_KEY"):
    _cohere_client = cohere.Client(__import__("os").environ["COHERE_API_KEY"])


def _format_chunks(docs, metas):
    return "\n\n".join(
        f"[Source: {m['title']} | video_id: {m['video_id']} at {m['start']:.0f}s]\n{d}"
        for d, m in zip(docs, metas)
    )


def retrieve_video_content(query: str) -> str:
    """
    Search the procurement/SCM video corpus using hybrid retrieval
    (embeddings + BM25 keyword search) and Cohere reranking when available,
    falling back to embeddings-only + a distance threshold otherwise. Use
    this for questions about procurement, sourcing, contracts, inventory,
    demand planning, logistics, risk, SRM, sustainability, or AI in
    procurement.
    """
    results = collection.query(
        query_embeddings=[embed_text(query)],
        n_results=EMBED_TOP_K,
        include=["documents", "metadatas", "distances"]
    )
    emb_docs = results["documents"][0]
    emb_metas = results["metadatas"][0]
    emb_dists = results["distances"][0]

    if _bm25 is None or _cohere_client is None:
        relevant = [
            (d, m) for d, m, dist in zip(emb_docs, emb_metas, emb_dists)
            if dist <= RELEVANCE_THRESHOLD
        ]
        if not relevant:
            return (
                "No sufficiently relevant content found in the video library "
                "for this query. Use general_procurement_knowledge instead."
            )
        docs, metas = zip(*relevant[:FINAL_TOP_K])
        return _format_chunks(docs, metas)

    # Hybrid: merge embedding candidates with BM25 candidates, dedupe by
    # (video_id, start), then rerank the merged set.
    tokenized_query = re.findall(r"\w+", query.lower())
    bm25_scores = _bm25.get_scores(tokenized_query)
    top_bm25_idx = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:BM25_TOP_K]

    candidates = {}
    for d, m in zip(emb_docs, emb_metas):
        candidates[(m["video_id"], round(m["start"], 1))] = (d, m)
    for i in top_bm25_idx:
        c = _all_chunks[i]
        key = (c["video_id"], round(c["start"], 1))
        candidates.setdefault(
            key,
            (c["text"], {"title": c["title"], "video_id": c["video_id"], "start": c["start"]})
        )

    candidate_list = list(candidates.values())
    if not candidate_list:
        return (
            "No sufficiently relevant content found in the video library "
            "for this query. Use general_procurement_knowledge instead."
        )

    docs_text = [d for d, _ in candidate_list]
    rerank_resp = _cohere_client.rerank(
        model="rerank-english-v3.0",
        query=query,
        documents=docs_text,
        top_n=min(FINAL_TOP_K, len(docs_text))
    )
    relevant = [
        candidate_list[r.index] for r in rerank_resp.results
        if r.relevance_score >= RERANK_THRESHOLD
    ]

    if not relevant:
        return (
            "No sufficiently relevant content found in the video library "
            "for this query. Use general_procurement_knowledge instead."
        )

    docs, metas = zip(*relevant)
    return _format_chunks(docs, metas)


video_retrieval_tool = Tool(
    name="video_content_search",
    func=retrieve_video_content,
    description=(
        "Searches a curated set of procurement and supply chain management "
        "video transcripts. Use this for any question about procurement, "
        "sourcing, contracts, inventory, demand planning, logistics, risk "
        "management, supplier relationships, sustainability, or AI in "
        "procurement. Returns relevant excerpts with source citations."
    )
)


## 13.5.1 Sanity check — confirm the tool works standalone

In [18]:
test_output = video_retrieval_tool.func("what are the stages of contract management?")
print(test_output)

[Source: Contract Management in Procurement | Stages & Tools | video_id: _l-rmPwyv28 at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used, Adobe Sign, CLM platforms for tracking execution. Stage 3. Contract moni

## 14 General knowledge Tool

A fallback for questions outside the video corpus — answers from the
LLM's own training knowledge, clearly labeled as such so users can tell
it apart from grounded, cited video-based answers.

In [19]:
def answer_general_knowledge(query: str) -> str:
    """
    Answer a general procurement/SCM knowledge question using the LLM's
    own training data, without retrieving from the video corpus. Used
    when the question falls outside the scope of the indexed videos.
    """
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a procurement and supply chain management "
                    "assistant. Answer using your general knowledge of "
                    "the field. Be clear, concise, and write in plain "
                    "prose rather than a numbered/bulleted breakdown "
                    "unless the question specifically asks for a list — "
                    "this keeps general-knowledge answers visually "
                    "distinct from video-sourced ones, which use "
                    "structured citations. Do not add your own "
                    "disclaimer about the source; the app adds that "
                    "label separately. If the question is entirely "
                    "unrelated to procurement or supply chain (e.g. "
                    "weather, recipes, general trivia), do not reframe "
                    "it into a procurement angle or partially answer "
                    "it — state plainly that it's outside scope and "
                    "invite a relevant question instead."
                )
            },
            {"role": "user", "content": query}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content


general_knowledge_tool = Tool(
    name="general_procurement_knowledge",
    func=answer_general_knowledge,
    description=(
        "Answers general procurement, sourcing, or supply chain "
        "management questions using broad domain knowledge, when the "
        "question is not covered by the indexed video library — for "
        "example, definitions, industry standards, certifications, or "
        "topics the videos don't address. Does not provide source "
        "citations from the videos."
    )
)


## 14.1 Sanity check — confirm the tool works standalone

In [20]:
test_output = general_knowledge_tool.func("What is ethical procurement per CIPS?")
print(test_output)

Ethical procurement, as defined by the Chartered Institute of Procurement & Supply (CIPS), refers to the practice of acquiring goods and services in a manner that is not only economically viable but also socially responsible and environmentally sustainable. This approach emphasizes the importance of considering the ethical implications of procurement decisions, including the treatment of workers, the impact on communities, and the environmental consequences of sourcing practices.

CIPS advocates for transparency, fairness, and integrity in procurement processes, encouraging organizations to engage with suppliers who uphold similar ethical standards. This includes ensuring that suppliers adhere to labor rights, avoid exploitation, and minimize environmental harm. By integrating ethical considerations into procurement strategies, organizations can contribute positively to society and the environment while also enhancing their reputation and stakeholder trust.


## 15. Basic QA chain

Takes a user question → retrieves top-k relevant chunks from Chroma →
passes them as context to an LLM → returns a grounded answer with
source citations (video title + timestamp).

In [21]:
def answer_question(question: str, n_results: int = 4) -> dict:
    """
    Retrieve relevant chunks and generate an answer grounded in them.

    Returns a dict with the answer text and the source chunks used,
    so citations can be shown alongside the response.
    """
    results = collection.query(
        query_embeddings=[embed_text(question)],
        n_results=n_results
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    context = "\n\n".join(
        f"[Source: {m['title']} at {m['start']:.0f}s]\n{d}"
        for d, m in zip(docs, metas)
    )

    system_prompt = (
        "You are a procurement and supply chain management assistant. "
        "Answer the user's question using ONLY the provided context. "
        "If the context doesn't contain enough information to answer, say so. "
        "Cite the video title when referencing specific information."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        temperature=0.2
    )

    return {
        "answer": response.choices[0].message.content,
        "sources": [
            {"title": m["title"], "start": m["start"], "video_id": m["video_id"]}
            for m in metas
        ]
    }

## 15.1 Test the full pipeline

In [22]:
result = answer_question("What are the four stages of contract management?")

print("ANSWER:")
print(result["answer"])
print("\nSOURCES:")
for s in result["sources"]:
    print(f"- {s['title']} ({s['start']:.0f}s) — youtube.com/watch?v={s['video_id']}&t={int(s['start'])}")

ANSWER:
The four stages of contract management are:

1. Contract creation and negotiation: Identify procurement needs and draft contract terms including deliverables, pricing, and timelines.
2. Contract execution: Finalize and sign agreements through digital or physical signatures and record contracts in a central repository.
3. Contract monitoring and compliance: Track SLAs, renewal dates, and non-compliance events, and run audits to check deliverables against timelines.
4. Contract renewal or closure: Evaluate supplier performance and determine whether to extend or terminate the contract, and archive closed contracts while collecting post-mortem lessons. 

This information is sourced from "Contract Management in Procurement | Stages & Tools."

SOURCES:
- Contract Management in Procurement | Stages & Tools (59s) — youtube.com/watch?v=_l-rmPwyv28&t=58
- Contract Management in Procurement | Stages & Tools (129s) — youtube.com/watch?v=_l-rmPwyv28&t=129
- Contract Management in Procuremen

## 16. Build the agent

Combines both tools with a decision-making LLM. The agent reads the
user's question, compares it against each tool's description, and
decides which to call — video retrieval, general knowledge, or both.

In [23]:
!pip install langchain-openai


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 16.1Enable langSmith Tracing

In [24]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "procurement-scm-assistant"

In [25]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = [video_retrieval_tool, general_knowledge_tool]

agent_system_prompt = agent_system_prompt = (
    "You are a procurement and supply chain management assistant. "
    "You have two tools available:\n"
    "- video_content_search: searches a curated video library covering "
    "procurement, sourcing, contracts, inventory, demand planning, "
    "logistics, risk, SRM, sustainability, AI in procurement\n"
    "- general_procurement_knowledge: general domain knowledge, not "
    "sourced from videos\n\n"
    "MANDATORY RULE: For every question, you MUST call "
    "video_content_search FIRST, before considering any other tool. "
    "Only call general_procurement_knowledge if video_content_search's "
    "results are empty, clearly irrelevant, or explicitly insufficient "
    "to answer the question. Never skip video_content_search."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", agent_system_prompt),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

## 16.2 Sanity check — test routing on both question types

In [26]:
# Should route to video_content_search
result1 = agent_executor.invoke({"input": "What are the stages of contract management?"})
print(result1["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `stages of contract management`


[Source: Contract Management in Procurement | Stages & Tools | video_id: _l-rmPwyv28 at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools used,

In [27]:
# Should route to general_procurement_knowledge
result2 = agent_executor.invoke({"input": "What is ethical procurement per CIPS?"})
print(result2["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `ethical procurement CIPS`


No sufficiently relevant content found in the video library for this query. Use general_procurement_knowledge instead.
Invoking: `general_procurement_knowledge` with `ethical procurement CIPS definition`


Ethical procurement, as defined by the Chartered Institute of Procurement & Supply (CIPS), refers to the process of acquiring goods and services in a manner that is not only efficient and cost-effective but also socially responsible and sustainable. This involves considering the ethical implications of purchasing decisions, such as the impact on the environment, labor practices, and the welfare of communities. Ethical procurement aims to ensure that suppliers adhere to fair labor standards, environmental regulations, and ethical business practices, ultimately contributing to a more sustainable and equitable supply chain.Ethical procurement, as defined by the Chartered Institute of Procurement & Supply (CIPS), involve

## 17 Sanity check — confirm traces are logging in Langsmith

In [28]:
result = agent_executor.invoke({"input": "What are the four components of a procurement contract?"})
print(result["output"])
print("\nCheck smith.langchain.com under project 'procurement-scm-assistant' to see this trace.")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `components of a procurement contract`


[Source: Contract Management in Procurement | Stages & Tools | video_id: _l-rmPwyv28 at 129s]
Stage 4. Contract renewal or closure. Evaluate supplier performance and determine whether to extend or terminate the contract. Archive closed contracts and collect postmorm lessons. Example, a software company uses a 12-month SaaS subscription for a CRM tool. Near expiration, procurement evaluates usage, cost effectiveness, and vendor support to decide on renewal. Four, components of a procurement contract. Scope of work, SOW clear definition of the goods or services to be delivered. Pricing and payment terms include structure, fixed, milestone, hourly, due dates, and penalties. Service level agreements, SLAs, and KPIs, define expectations for delivery times, quality, and issue resolution. Termination clauses, outline under what circumstances the contract may be terminated early. Force majeure, provision to manage 

## 18. Add conversational memory to the agent

Wraps agent_executor calls with a running chat history so follow-up
questions ("what about the second one?") resolve correctly without the
user re-stating context. Uses LangChain's message objects to keep the
history in the format the agent's prompt template expects.

In [29]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def ask_agent(question: str) -> dict:
    """
    Invoke the agent with the running chat history, then update history
    with this turn's question and answer.
    """
    result = agent_executor.invoke({
        "input": question,
        "chat_history": chat_history
    })

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["output"]))

    return result

## 18.1 Sanity check — test memory with a follow-up

In [30]:
r1 = ask_agent("What are the four stages of contract management?")
print(r1["output"])

print("\n---\n")



Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `four stages of contract management`


[Source: Contract Management in Procurement | Stages & Tools | video_id: _l-rmPwyv28 at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository. Tools 

In [31]:
r2 = ask_agent("Can you explain the second one in more detail?")
print(r2["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_content_search` with `contract execution in contract management`


[Source: Contract Management in Procurement | Stages & Tools | video_id: _l-rmPwyv28 at 59s]
Compliance and risk mitigation ensures regulatory, legal, and internal compliance. Cost control prevents maverick spending and ensures price adherence. Performance tracking monitors supplier adherence to KPIs and SLAs. Dispute resolution provides a formal mechanism to resolve disagreements. 3. Key stages of contract management. Stage 1. Contract creation and negotiation. Identify procurement needs and draft contract terms including deliverables, pricing, and timelines. Collaborate with legal and finance teams for term approvals. Tools used, Microsoft Word for drafting, DocuSign for review workflows, Ironclad for negotiation version tracking. Stage 2. Contract execution. Finalize and sign agreements through digital or physical signatures. Route for internal approvals and record contracts in a central repository.

## 19. Video metadata Tool

Answers questions about the video library itself — what's covered,
how many videos, which titles exist — without doing semantic search
over chunk content. Useful for questions like "what topics do you
cover?" or "how many videos are in your library?"

In [32]:
def get_video_metadata(query: str) -> str:
    """
    Answer questions about the video library's contents — titles, topic
    coverage, video count — without searching chunk text. Builds its
    answer directly from the loaded transcript metadata.
    """
    video_list = []
    for vid, content in data.items():
        video_list.append(f"- {content['title']} (video_id: {vid})")

    summary = (
        f"The library contains {len(data)} videos:\n"
        + "\n".join(video_list)
    )
    return summary


video_metadata_tool = Tool(
    name="video_library_metadata",
    func=get_video_metadata,
    description=(
        "Answers questions about the video library's contents as a "
        "whole — for example, how many videos are indexed, what topics "
        "or titles are covered, or which video discusses a specific "
        "subject at a high level. Does NOT search inside video "
        "transcripts — use video_content_search for that instead."
    )
)

## 19.1 Sanity check — confirm the tool works standalone

In [33]:
test_output = video_metadata_tool.func("what topics are covered?")
print(test_output)

The library contains 10 videos:
- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (video_id: uSjrTpJHn2g)
- Contract Management in Procurement | Stages & Tools (video_id: _l-rmPwyv28)
- Demand Planning Explained | Process, Benefits & Forecasting (video_id: W10PJUFTH9M)
- An Introduction to Logistics and Supply Chain Management (video_id: GCK18JyPVXI)
- Digital Transformation & AI in Procurement (video_id: R297gX6-KvQ)
- Sustainable Procurement Explained: Key Frameworks and Benefits (video_id: U1E6rtnreoA)
- What is Supplier Relationship Management? Supply Chain 101 (video_id: dswoMsZwRuo)
- Supply Chain Risk Management Strategy | Response to Risk | Contingency Plan (video_id: ymFVfng3dXE)
- What is Procurement? Procurement Process Explained in 12 minutes (video_id: ZCIJQJRw6xw)
- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (video_id: 0ZDrpf5aMiw)


## 19.2 Add the tool to the agent

In [34]:
tools = [video_retrieval_tool, general_knowledge_tool, video_metadata_tool]

agent_system_prompt = (
    "You are a procurement and supply chain management assistant. "
    "You have three tools available:\n"
    "- video_content_search: searches inside the curated video library "
    "for specific procurement/SCM content\n"
    "- video_library_metadata: answers questions about the library "
    "itself — titles, topic coverage, video count\n"
    "- general_procurement_knowledge: general domain knowledge, not "
    "sourced from videos\n\n"
    "For most questions, a video library search has ALREADY been run "
    "for you and the results are included at the top of the human "
    "message, labeled 'Video library search results'. Use them "
    "directly if they answer the question — you do NOT need to call "
    "video_content_search again for the same question. Only call it "
    "yourself if you need a meaningfully different or more specific "
    "search.\n\n"
    "Routing:\n"
    "1. If the question asks what the library contains, covers, or how "
    "many videos it has, call video_library_metadata.\n"
    "2. If the provided search results say 'No sufficiently relevant "
    "content found', or the question asks something those results "
    "don't cover — including general definitions, standards, or "
    "certifications (e.g. CIPS) — call general_procurement_knowledge "
    "for that part.\n"
    "3. A single question can need more than one tool — e.g. one part "
    "answerable from the provided video results and another part that "
    "isn't. Call whatever combination of tools is needed to fully "
    "answer every part of the question; don't drop a part just "
    "because another part was already covered.\n"
    "4. If a question is entirely outside procurement/supply chain, "
    "call general_procurement_knowledge and let it explain that the "
    "topic is out of scope.\n"
    "5. When you use the video search results — whether pre-provided "
    "or from your own video_content_search call — answer using ONLY "
    "what those excerpts actually state. Do not add examples, tools, "
    "steps, or facts from your own general knowledge, even ones that "
    "seem obviously true or fit naturally alongside what's retrieved "
    "— a plausible addition is not something you actually know is in "
    "THIS video. If the excerpts don't fully answer the question, say "
    "so explicitly rather than filling the gap yourself; call "
    "general_procurement_knowledge for that instead, and make clear "
    "in your answer which parts came from the video versus general "
    "knowledge."
)

prompt = ChatPromptTemplate.from_messages([
    ("system", agent_system_prompt),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, return_intermediate_steps=True)


In [35]:
# --- Retrieve-first wrapper ---
# gpt-4o-mini sometimes skips calling video_content_search entirely on
# questions that sound like a "well-known" business framework (e.g. "6
# steps in the strategic sourcing flywheel"), answering from its own
# training data instead — even with the "CRITICAL RULE" this notebook
# used to have telling it to search first. A stronger prompt doesn't
# reliably fix a model discretion problem, so retrieval is no longer
# optional: it always runs first in code, before the agent gets a chance
# to decide whether to bother.
#
# First version of this wrapper bypassed the agent entirely whenever
# retrieval found something, which fixed that bug but broke compound
# questions (a two-part question would only get the video half answered,
# since bypassing the agent meant general_procurement_knowledge never got
# a chance to run for the other part). This version always runs the
# agent, with retrieval results already attached to its input — so it
# can't skip searching, but can still reach for a second tool when a
# question has a part the video results don't cover.

class _FakeTool:
    """Lets the forced-retrieval path report tools_used the same way
    AgentExecutor's real intermediate_steps do (step[0].tool)."""
    def __init__(self, tool):
        self.tool = tool


def answer_question(question: str, chat_history: list) -> dict:
    retrieved = retrieve_video_content(question)
    augmented_input = (
        f"Video library search results for this question:\n{retrieved}\n\n"
        f"Question: {question}"
    )
    result = agent_executor.invoke({"input": augmented_input, "chat_history": chat_history})

    steps = result.get("intermediate_steps", [])
    tools_used = [s[0].tool for s in steps]
    if (
        not retrieved.startswith("No sufficiently relevant content found")
        and "video_content_search" not in tools_used
    ):
        steps = [(_FakeTool("video_content_search"), retrieved)] + steps

    return {"output": result["output"], "intermediate_steps": steps}


# Redefine ask_agent (originally from section 16) to route through the
# retrieve-first wrapper instead of calling agent_executor directly, now
# that video_content_search is a forced pre-step rather than an
# agent-chosen tool. Every cell below that calls ask_agent(...) picks up
# this version automatically.
def ask_agent(question: str) -> dict:
    """
    Answer via the retrieve-first wrapper, using the running chat history,
    then update history with this turn's question and answer.
    """
    result = answer_question(question, chat_history)

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=result["output"]))

    return result


## 19.3 Sanity check — test metadata routing through the agent

In [36]:
result = ask_agent("How many videos are in your library, and what topics do they cover?")
print(result["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `video_library_metadata` with `number of videos in the library`


The library contains 10 videos:
- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (video_id: uSjrTpJHn2g)
- Contract Management in Procurement | Stages & Tools (video_id: _l-rmPwyv28)
- Demand Planning Explained | Process, Benefits & Forecasting (video_id: W10PJUFTH9M)
- An Introduction to Logistics and Supply Chain Management (video_id: GCK18JyPVXI)
- Digital Transformation & AI in Procurement (video_id: R297gX6-KvQ)
- Sustainable Procurement Explained: Key Frameworks and Benefits (video_id: U1E6rtnreoA)
- What is Supplier Relationship Management? Supply Chain 101 (video_id: dswoMsZwRuo)
- Supply Chain Risk Management Strategy | Response to Risk | Contingency Plan (video_id: ymFVfng3dXE)
- What is Procurement? Procurement Process Explained in 12 minutes (video_id: ZCIJQJRw6xw)
- Understanding Inventory Management (Inside the Supply Chain Series) Lesson 1 (video_id: 0ZDrpf5aMiw)

## 20. Adding a voice input tool

In [37]:
pip install openai sounddevice scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
def transcribe_audio(audio_file) -> str:
    """Transcribe an audio file to text using OpenAI Whisper API."""
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )
    return transcript.text

## 20.1 Voice input sanity check

In [39]:
import sounddevice as sd
from scipy.io.wavfile import write

fs = 44100
duration = 5  # seconds
print("Recording...")
recording = sd.rec(int(duration * fs), samplerate=fs, channels=1)
sd.wait()
write("test_voice.wav", fs, recording)
print("Done recording.")

text = transcribe_audio(open("test_voice.wav", "rb"))
print("Transcribed text:", text)

result = ask_agent(text)
print(result["output"])

Recording...
Done recording.


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Transcribed text: Thank you for watching.

Invoking: `general_procurement_knowledge` with `What is the significance of thanking someone after a presentation or video?`


Thanking someone after a presentation or video is significant for several reasons. It shows appreciation for the audience's time and attention, which helps build rapport and fosters a positive relationship. Acknowledging their presence also reinforces the importance of their participation and feedback, making them feel valued. Additionally, expressing gratitude can enhance the overall atmosphere of the interaction, encouraging open communication and engagement in future discussions. This practice contributes to a respectful and professional environment, which is essential in both personal and business contexts.Thanking someone after a presentation or video is significant for several reasons:

1. **Appreciation**: It shows appreciation for the audience's time and attention, helping to build rapport and foster a positive

In [40]:
test_queries = [
    "What does the video about contract management say, and how many videos do you have total?",  # forces multi-tool use
    "Tell me about supplier risk management",  # could hit RAG content tool OR general_knowledge — ambiguous on purpose
    "What did I just ask you?",  # should hit memory, not RAG or metadata
    "How many videos discuss sustainability, and briefly explain what sustainable procurement means",  # metadata + general knowledge combo
    "What's the capital of France?",  # should NOT match any procurement tool — tests fallback/refusal behavior
    "Summarize everything you know about demand planning from the video library",  # should hit RAG, not general_knowledge, since "from the video library" is explicit
]

for q in test_queries:
    print(f"\n{'='*60}\nQUERY: {q}\n{'='*60}")
    result = ask_agent(q)
    print(result["output"])


QUERY: What does the video about contract management say, and how many videos do you have total?


Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `general_procurement_knowledge` with `What does the video about contract management say?`


I'm unable to provide details about specific videos. However, I can offer a general overview of contract management in procurement and supply chain management. Contract management involves the process of creating, executing, and monitoring contracts to ensure compliance and performance. It includes negotiating terms, managing relationships with suppliers, and ensuring that all parties fulfill their obligations. Effective contract management helps mitigate risks, control costs, and improve supplier performance. If you have specific questions about contract management, feel free to ask!
Invoking: `video_library_metadata` with `How many videos are in the library?`


The library contains 10 videos:
- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (video_id: uSjrTpJHn2g)
- Contract Management in Procurement | Stages & Tools (video_id: _l-rmPwyv28)
- Demand 

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The video about contract management in procurement discusses the process of creating, executing, and monitoring contracts to ensure compliance and performance. It includes aspects such as negotiating terms, managing relationships with suppliers, and ensuring that all parties fulfill their obligations. Effective contract management is crucial for mitigating risks, controlling costs, and improving supplier performance.

Additionally, the library contains a total of **10 videos** covering various topics related to procurement and supply chain management. Here are the titles:

1. **6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results**
2. **Contract Management in Procurement | Stages & Tools**
3. **Demand Planning Explained | Process, Benefits & Forecasting**
4. **An Introduction to Logistics and Supply Chain Management**
5. **Digital Transformation & AI in Procurement**
6. **Sustainable Procurement Explained: Key Frameworks and Benefits**
7. **What is Supplier R

...


Invoking: `video_content_search` with `sustainability`


No sufficiently relevant content found in the video library for this query. Use general_procurement_knowledge instead.
Invoking: `video_library_metadata` with `total videos`


The library contains 10 videos:
- 6-Step Strategic Sourcing Flywheel: Turning Supplier Relationships into Results (video_id: uSjrTpJHn2g)
- Contract Management in Procurement | Stages & Tools (video_id: _l-rmPwyv28)
- Demand Planning Explained | Process, Benefits & Forecasting (video_id: W10PJUFTH9M)
- An Introduction to Logistics and Supply Chain Management (video_id: GCK18JyPVXI)
- Digital Transformation & AI in Procurement (video_id: R297gX6-KvQ)
- Sustainable Procurement Explained: Key Frameworks and Benefits (video_id: U1E6rtnreoA)
- What is Supplier Relationship Management? Supply Chain 101 (video_id: dswoMsZwRuo)
- Supply Chain Risk Management Strategy | Response to Risk | Contingency Plan (video_id: ymFVfng3dXE)
- What is Procurement? Procurement P

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


In the video library, there is **one video** that discusses sustainability, titled **"Sustainable Procurement Explained: Key Frameworks and Benefits."**

### What is Sustainable Procurement?
Sustainable procurement refers to the process of acquiring goods and services while considering not only economic factors but also the environmental and social impacts of those purchases. The goal is to minimize negative effects on the environment and society while maximizing positive outcomes.

In practice, sustainable procurement involves selecting suppliers and products that are environmentally friendly, ethically produced, and socially responsible. This includes considerations such as:
- Reducing carbon footprints
- Ensuring fair labor practices
- Promoting the use of renewable resources

Organizations that adopt sustainable procurement practices aim to align their purchasing decisions with broader sustainability goals, such as reducing waste, supporting local economies, and enhancing corporate

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


The question about the capital of France is outside the scope of procurement and supply chain management. If you have any questions related to those fields, feel free to ask!

> Finished chain.
The question about the capital of France is outside the scope of procurement and supply chain management. If you have any questions related to those fields, feel free to ask!

QUERY: Summarize everything you know about demand planning from the video library
The video titled **"Demand Planning Explained | Process, Benefits & Forecasting"** provides a comprehensive overview of demand planning, which is the process of forecasting the demand for a product or service to ensure efficient production and timely delivery to customers. Here are the key points summarized from the video:

### Definition and Purpose:
- **Demand Planning**: It involves forecasting future demand to balance inventory levels, ensuring sufficient stock to meet customer needs without excess inventory.
- The process typically cover